In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

    !pip install -q kaggle mlflow wandb
    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn
import wandb

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [ ]:
SEQ_LEN = 52
KERNEL_SIZE = 25

class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        left = (self.kernel_size - 1) // 2
        right = self.kernel_size - 1 - left
        front = x[:, 0:1].repeat(1, left)
        end = x[:, -1:].repeat(1, right)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.unsqueeze(1)).squeeze(1)

class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len=1, kernel_size=25):
        super().__init__()
        self.decomp = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        trend = self.decomp(x)
        seasonal = x - trend
        return self.linear_trend(trend) + self.linear_seasonal(seasonal)

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, windows, targets, weights):
        self.windows = windows
        self.targets = targets
        self.weights = weights

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx], self.weights[idx]

def build_series_matrix(df):
    pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales')
    pivot = pivot.sort_index(axis=1)
    holiday = df.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()
    holiday = holiday.reindex(pivot.columns).fillna(False)
    return pivot, holiday

def make_windows(pivot, holiday, seq_len=SEQ_LEN):
    values = pivot.fillna(0).values
    n_series, n_time = values.shape

    windows, targets, weights = [], [], []
    for t in range(seq_len, n_time):
        window = values[:, t - seq_len:t]
        target = values[:, t]
        valid = ~pivot.iloc[:, t].isna().values
        if valid.sum() == 0:
            continue
        w = np.where(holiday.iloc[t], 5.0, 1.0)
        windows.append(window[valid])
        targets.append(target[valid])
        weights.append(np.full(valid.sum(), w))
    return np.concatenate(windows), np.concatenate(targets), np.concatenate(weights)

In [ ]:
class DLinearForecaster(BaseEstimator):
    def __init__(self, seq_len=52, kernel_size=25, epochs=15, lr=1e-3, batch_size=512):
        self.seq_len = seq_len
        self.kernel_size = kernel_size
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size

    def fit(self, X, y, log_fn=None):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        pivot, holiday = build_series_matrix(df)
        self.series_mean_ = pivot.mean(axis=1).fillna(0)
        self.series_std_ = pivot.std(axis=1).fillna(1).replace(0, 1)

        norm_pivot = pivot.sub(self.series_mean_, axis=0).div(self.series_std_, axis=0)
        windows, targets, weights = make_windows(norm_pivot, holiday, self.seq_len)

        X_t = torch.tensor(windows, dtype=torch.float32)
        y_t = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        w_t = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        loader = DataLoader(WindowDataset(X_t, y_t, w_t), batch_size=self.batch_size, shuffle=True)

        self.model_ = DLinear(self.seq_len, 1, self.kernel_size)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for xb, yb, wb in loader:
                opt.zero_grad()
                pred = self.model_(xb)
                loss = (wb * (pred - yb).abs()).mean()
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
            epoch_loss /= len(loader.dataset)
            if log_fn is not None:
                log_fn(epoch, epoch_loss)

        self.history_ = pivot
        return self

    def _forecast_series(self, key, target_dates):
        if key not in self.history_.index:
            return pd.Series(self.series_mean_.mean(), index=target_dates)

        mean = self.series_mean_.loc[key]
        std = self.series_std_.loc[key]
        series = self.history_.loc[key].copy()

        max_date = max(target_dates.max(), series.index.max())
        all_dates = pd.date_range(series.index.min(), max_date, freq='W-FRI')

        series = series.reindex(all_dates)
        known_mask = series.notna()
        series = series.fillna(0)

        values = ((series - mean) / std).values.tolist()

        self.model_.eval()
        with torch.no_grad():
            for i in range(len(values)):
                if i >= self.seq_len and not known_mask.iloc[i]:
                    window = torch.tensor([values[i - self.seq_len:i]], dtype=torch.float32)
                    values[i] = self.model_(window).item()

        result = pd.Series(values, index=all_dates) * std + mean
        return result.reindex(target_dates)

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        preds = np.zeros(len(df))
        for (store, dept), group in df.groupby(['Store', 'Dept']):
            forecast = self._forecast_series((store, dept), pd.DatetimeIndex(group['Date'].unique()))
            preds[group.index] = forecast.reindex(group['Date']).values
        return preds

In [ ]:
mlflow.set_experiment('DLinear_Training')

In [ ]:
with mlflow.start_run(run_name='DLinear_Cleaning'):
    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')

    y_train = train['Weekly_Sales'].clip(lower=0)

In [ ]:
with mlflow.start_run(run_name='DLinear_Windowing'):
    mlflow.log_param('seq_len', SEQ_LEN)
    mlflow.log_param('kernel_size', KERNEL_SIZE)

    pivot, holiday = build_series_matrix(train.assign(Weekly_Sales=y_train))
    windows, targets, weights = make_windows(pivot, holiday, SEQ_LEN)
    mlflow.log_metric('n_windows', len(windows))
    mlflow.log_metric('n_series', len(pivot))

len(windows), pivot.shape

In [ ]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

In [ ]:
params = dict(seq_len=SEQ_LEN, kernel_size=KERNEL_SIZE, epochs=15, lr=1e-3, batch_size=512)

with mlflow.start_run(run_name='DLinear_CV'):
    mlflow.log_params(params)

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        wandb.init(project='walmart-sales-forecasting', name=f'DLinear_CV_fold{i}', config=params, reinit='finish_previous')

        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        def log_fn(epoch, loss, fold=i):
            mlflow.log_metric(f'train_loss_fold{fold}', loss, step=epoch)
            wandb.log({'epoch': epoch, 'train_loss': loss})

        model = DLinearForecaster(**params)
        model.fit(train.loc[train_mask, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[train_mask], log_fn=log_fn)

        preds = model.predict(train.loc[val_mask, ['Store', 'Dept', 'Date', 'IsHoliday']])
        score = wmae(y_train[val_mask].values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)

        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'val_wmae': score})
        wandb.finish()

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))

fold_scores

In [ ]:
with mlflow.start_run(run_name='DLinear_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='DLinear_Final', config=params, reinit='finish_previous')

    def log_fn(epoch, loss):
        mlflow.log_metric('train_loss', loss, step=epoch)
        wandb.log({'epoch': epoch, 'train_loss': loss})

    pipeline = Pipeline([
        ('model', DLinearForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train, model__log_fn=log_fn)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae